In [1]:
import polars as pl

### Load the datasets:
- ADA - Area of Activity (professional occupation)
- Attivita - Skills related to Area of Activity (many to one)
- CP2021 - Classification of Professions (more granular than Areas of Activity)

In [2]:
ada = pl.read_csv("../data/rtc/ada_cp2021.csv", separator=";")
attivita = pl.read_csv("../data/rtc/attivita_ada.csv", separator=";")
cp2021 = pl.read_csv("../data/rtc/five_digit-cp2021.csv", separator=";").select(pl.col(["codice_cp2021", "cp2021_name", "cp2021_desc"]))

# Convert column "sep" to string
ada = ada.with_columns(pl.col("sep").cast(pl.String))
attivita = attivita.with_columns(pl.col("sep").cast(pl.String))

print(f"Number of rows: {ada.height:,} (number of Areas of Activity (ADA))")
print(f"Number of rows: {attivita.height:,} (number of Activities (Skills))")

Number of rows: 1,859 (number of Areas of Activity (ADA))
Number of rows: 7,319 (number of Activities (Skills))


In [3]:
ada = ada.join(cp2021, on='codice_cp2021', how='left')
del cp2021

print(f"Number of rows: {ada.height:,} (number of Areas of Activity (ADA) after merging with CP2021)")

Number of rows: 1,859 (number of Areas of Activity (ADA) after merging with CP2021)


In [6]:
ada_and_attivita_flat = attivita.join(ada, on='codice_ada', how='left')
ada_and_attivita_flat = ada_and_attivita_flat.drop("sep_right")

print(f"Number of rows: {ada_and_attivita_flat.height:,} (number of skills mentioned in ADA)")

Number of rows: 14,395 (number of skills mentioned in ADA)


In [8]:
# Transform the ada_attivita df to create nested dataframe of skills for each ADA

ada_and_attivita_nested_cp = ada_and_attivita_flat.group_by("id_attivita", maintain_order=True).agg([
    pl.struct([
        "codice_cp2021", 
        "cp2021_name",
        "cp2021_desc"
    ]).alias("cp2021"),
    pl.all().exclude([
        "codice_cp2021", 
        "cp2021_name",
        "cp2021_desc",
    ]).first()
])

print(f"Number of rows: {ada_and_attivita_nested_cp.height:,} (number of attivita entries with nested cp2021)")

Number of rows: 7,319 (number of attivita entries with nested cp2021)


In [10]:
# Transform the df to create a deeper level of nested cp2021 occupations and skills for each ADA

ada_and_attivita_nested = ada_and_attivita_nested_cp.group_by("codice_ada", maintain_order=True).agg([
    pl.struct([
        "id_attivita", 
        "attivita",
        "cp2021"
    ]).alias("attivita"),
    pl.all().exclude([
        "id_attivita", 
        "attivita",
        "cp2021"
    ]).first()
])

print(f"Number of rows: {ada_and_attivita_nested.height:,} (number of ADA entries with nested attivita and cp2021)")

Number of rows: 961 (number of ADA entries with nested attivita and cp2021)


In [14]:
ada_and_attivita_nested.write_parquet("../data/rtc/ada_and_attivita_nested.parquet")

In [15]:
# Inspect the dataframe for a specific ADA
ada_to_find = "ADA.01.01.01"

# Filter for the ID, explode the list, then unnest the struct
skills_unnested = (ada_and_attivita_nested
    .filter(pl.col("codice_ada") == ada_to_find)
    .select("attivita") # optionally add "codice_ada"
    .explode("attivita")
    .unnest("attivita")
    # .explode("cp2021")
    # .unnest("cp2021")
)
print(f"Number of skills for this ADA: {skills_unnested.height}\n")
skills_unnested

Number of skills for this ADA: 12



id_attivita,attivita,cp2021
i64,str,list[struct[3]]
8049,"""Valutazione dell'idoneità ambi…","[{""2.3.1.3.0"",""Agronomi e forestali"",""Le professioni comprese in questa unità conducono ricerche ovvero applicano le conoscenze esistenti nel campo della cura e dell'allevamento di animali e di vegetali. Studiano le modalità riproduttive, la genetica e le possibilità di miglioramento delle specie, i fattori di crescita e nutrizionali degli animali da allevamento, delle piante e delle colture; la composizione chimica, fisica, biologica e minerale dei suoli, individuando le colture più adattabili e a maggiore rendimento; ricercano e mettono a punto nuove pratiche e modalità colturali e di allevamento; studiano, identificano e controllano le malattie dei vegetali, ne individuano le modalità di trattamento sia chimico che biologico. Definiscono le modalità di gestione, di miglioramento, di protezione delle risorse floro-faunistiche naturali; della loro messa a produzione; di salvaguardia dell'idrologia, della qualità delle acque e della stabilità del suolo e di ripopolamento del loro habitat naturale. L'esercizio delle professioni di Dottore Agronomo e di Dottore Forestale è regolato dalle leggi dello Stato.""}, {""2.3.1.1.5"",""Botanici"",""Le professioni comprese in questa unità conducono ricerche su concetti e teorie fondamentali nel campo della botanica, incrementano la conoscenza scientifica in materia e la applicano in attività di ricerca e nelle sperimentazioni di laboratorio. Studiano le forme e le origini della vita vegetale, la genetica, i processi vitali, le malattie e le modalità di sviluppo e di evoluzione. Applicano e rendono disponibili tali conoscenze nella produzione di beni e servizi.""}, {""3.2.2.1.1"",""Tecnici agronomi"",""Le professioni comprese in questa unità assistono gli specialisti ovvero eseguono procedure e tecniche proprie nella progettazione di sistemi agricoli, agroalimentari e zootecnici, nel miglioramento delle colture e delle relative condizioni di crescita e di difesa, nell'individuazione delle colture piu adattabili e piu redditizie, nell'individuazione e nel controllo delle malattie dei vegetali, nella conservazione della biodiversita colturale.""}]"
8050,"""Indagine della composizione de…","[{""2.3.1.3.0"",""Agronomi e forestali"",""Le professioni comprese in questa unità conducono ricerche ovvero applicano le conoscenze esistenti nel campo della cura e dell'allevamento di animali e di vegetali. Studiano le modalità riproduttive, la genetica e le possibilità di miglioramento delle specie, i fattori di crescita e nutrizionali degli animali da allevamento, delle piante e delle colture; la composizione chimica, fisica, biologica e minerale dei suoli, individuando le colture più adattabili e a maggiore rendimento; ricercano e mettono a punto nuove pratiche e modalità colturali e di allevamento; studiano, identificano e controllano le malattie dei vegetali, ne individuano le modalità di trattamento sia chimico che biologico. Definiscono le modalità di gestione, di miglioramento, di protezione delle risorse floro-faunistiche naturali; della loro messa a produzione; di salvaguardia dell'idrologia, della qualità delle acque e della stabilità del suolo e di ripopolamento del loro habitat naturale. L'esercizio delle professioni di Dottore Agronomo e di Dottore Forestale è regolato dalle leggi dello Stato.""}, {""2.3.1.1.5"",""Botanici"",""Le professioni comprese in questa unità conducono ricerche su concetti e teorie fondamentali nel campo della botanica, incrementano la conoscenza scientifica in materia e la applicano in attività di ricerca e nelle sperimentazioni di laboratorio. Studiano le forme e le origini della vita vegetale, la genetica, i processi vitali, le malattie e le modalità di sviluppo e di evoluzione. Applicano e rendono disponibili tali conoscenze nella produzione di beni e servizi.""}, {""3.2.2.1.1"",""Tecnici agronomi"",""Le professioni comprese in questa unità assistono gli specialisti